In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
let divide (numerator: float) (denominator: float) =
    if denominator = 0.0 then failwith "Division by zero"
    numerator / denominator

# Symbolic Differentiation 

In this notebook our goal is to implement *symbolic differentiation*.  Concretely, we want do implement a function `diff` that takes one argument:
  - The argument `expr` represents an *arithmetic expression*.
    Here an arithmetic expression is any string that is build from variable and numbers by application
    of any of the operator symbols "`+`", "`-`", "`*`", "`/`", and "`**`".
    The operator "`**`" represents exponentiation, i.e. an expression of the form 
    $a \texttt{**} b$ is interpreted as $a^b$.          
    Furthermore, if $e$ is an expression, then both $\exp(e)$ and $\ln(e)$ are expressions too.

The function call `diff(expr)` will then compute the derivative of `expr` with respect to the variable `x`.  For example, the function call 
`diff("x * exp(x)")` will compute the output
`1 * exp(x) + x * exp(x)` because we have:
$$ \frac{\mathrm{d}\;}{\mathrm{d}x} \bigl( x \cdot \mathrm{e}^x \bigr) = 1 \cdot \mathrm{e}^x + x \cdot \mathrm{e}^x. $$

This file shows the implementation of a program that can perform *symbolic differentiation* using `FsLexYacc`.  The grammar for the language implemented by this parser is as follows:
$$
\begin{array}{lcl}
  \texttt{expr}    & \rightarrow & \;\texttt{expr}\; \texttt{'+'} \; \texttt{product}  \\
                   & \mid        & \;\texttt{expr}\; \texttt{'-'} \; \texttt{product}  \\
                   & \mid        & \;\texttt{product}                                  \\[0.2cm]
  \texttt{product} & \rightarrow & \;\texttt{product}\; \texttt{'*'} \;\texttt{factor} \\
                   & \mid        & \;\texttt{product}\; \texttt{'/'} \;\texttt{factor} \\
                   & \mid        & \;\texttt{factor}                                   \\[0.2cm]
  \texttt{factor}  & \rightarrow & \texttt{base} \;\texttt{'**'} \; \texttt{factor}    \\
                   & \mid        & \texttt{base}                                       \\[0.2cm]
  \texttt{base}    & \rightarrow & \texttt{'exp'} \; \texttt{'('} \; \texttt{expr} \;\texttt{')'}     \\
                   & \mid        & \texttt{'ln'} \; \texttt{'('} \; \texttt{expr} \;\texttt{')'}      \\
                   & \mid        & \texttt{'('} \; \texttt{expr} \;\texttt{')'}                     \\
                   & \mid        & \;\texttt{NUMBER}                                                \\
                   & \mid        & \;\texttt{'x'}                               
\end{array}
$$

## Specification of the Scanner

In [ ]:
#r "SymbolicSyntax/bin/Debug/net9.0/FsLexYacc.Runtime.dll"
#r "SymbolicSyntax/bin/Debug/net9.0/SymbolicSyntax.dll"
open SymbolicSyntax

FsYacc declares token cases and their payload types; FsLex recognizes their spellings.

The scanner source is a real FsLex specification.

```fsharp
{
open SymbolicSyntax.Parser
open FSharp.Text.Lexing
let lexeme (buffer: LexBuffer<char>) = System.String(buffer.Lexeme)
}
let natural = '0' | ['1'-'9'] ['0'-'9']*
rule token = parse
 | [' ' '\t' '\r']+ { token lexbuf }
 | '\n' { lexbuf.EndPos <- lexbuf.EndPos.NextLine; token lexbuf }
 | natural { NUMBER(int (lexeme lexbuf)) }
 | "exp" { EXP } | "ln" { LN } | 'x' { X }
 | "**" { POWER } | '+' { PLUS } | '-' { MINUS } | '*' { TIMES } | '/' { DIVIDE }
 | '(' { LPAREN } | ')' { RPAREN }
 | eof { EOF }
 | _ { failwithf "Illegal character at line %d, column %d" lexbuf.StartPos.Line (lexbuf.StartPos.Column + 1) }

```

The token `NUMBER` specifies a *natural number*.

Below, we need to escape the meta charater `*`.

Blanks and tabulators are ignored.

Unkown characters are reported as lexical errors.

FsLexYacc specifications are compiled into an ordinary .NET assembly, which the notebook references.

The project build generates the lexer.

## Specification of the Parser

The start production is `expression`, which requires the whole input followed by EOF.

The FsYacc specification constructs a typed AST, with right-associative exponentiation.

```fsharp
%{
open SymbolicSyntax
%}
%token <int> NUMBER
%token X EXP LN PLUS MINUS TIMES DIVIDE POWER LPAREN RPAREN EOF
%start expression
%type <SymbolicSyntax.Expr> expression
%%
expression: expr EOF { $1 }
expr: expr PLUS product { Binary(Add, $1, $3) }
    | expr MINUS product { Binary(Subtract, $1, $3) }
    | product { $1 }
product: product TIMES factor { Binary(Multiply, $1, $3) }
       | product DIVIDE factor { Binary(Divide, $1, $3) }
       | factor { $1 }
factor: baseExpr POWER factor { Binary(Power, $1, $3) }
      | baseExpr { $1 }
baseExpr: EXP LPAREN expr RPAREN { Unary(Exp, $3) }
        | LN LPAREN expr RPAREN { Unary(Ln, $3) }
        | LPAREN expr RPAREN { $2 }
        | NUMBER { Number $1 }
        | X { Variable }

```

An *expr* is a sequence of *products* that are combined with the operator `+`.
The corresponding grammar rules are:
```
    expr : expr '+' product
         | expr '-' product
         | product
         ;
```

A *product* is a sequence of *factors* that are combined with the operator `*`.
The corresponding grammar rules are:
```
    product : product '*' factor
            | product '/' factor
            | factor
            ;
```

A `factor` is either a `base` expression raised to a power or it is just a `base` expression.
```
   factor : base '**' factor
          | base
          ;
```

A `base` expression can be the application of a function, an expression in parenthesis, a number, or the variable `x`.
```
    base : exp '(' expr ')'
         | ln  '(' expr ')'
         | '(' expr ')'
         | NUMBER
         | 'x'
         ;
```

The parse wrapper reports the line and column of invalid or incomplete input.

`Parse.expression source` invokes the generated scanner and parser.

In [ ]:
let parse = Parse.expression

In [ ]:
printfn "%A" (parse "ln(x ** x) + exp(x * x)")

Now we are ready to implement the function `diff` that takes an expression possibly containing the variable `x`.
It computes the derivative of the given expression with respect to `x`.
<ol>
<li>  The The corresponding cases implement the rule: 
      $$\frac{\mathrm{d}\;}{\mathrm{d}x}\bigl(f(x) + g(x)\bigr) = \frac{\mathrm{d}\;}{\mathrm{d}x} f(x) + \frac{\mathrm{d}\;}{\mathrm{d}x} g(x)$$
      </li>
<li>  The corresponding cases implement the rule:
      $$\frac{\mathrm{d}\;}{\mathrm{d}x}\bigl(f(x) - g(x)\bigr) = \frac{\mathrm{d}\;}{\mathrm{d}x} f(x) - \frac{\mathrm{d}\;}{\mathrm{d}x} g(x)$$
      </li>
<li>  The corresponding cases deals with the case where <tt>e</tt> is a product.  The 
      <a href="https://en.wikipedia.org/wiki/Product_rule">product rule</a> is      
      $$ \frac{\mathrm{d}\;}{\mathrm{d}x}\bigl(f(x) \cdot g(x)\bigr) = \left(\frac{\mathrm{d}\;}{\mathrm{d}x} f(x)\right)\cdot g(x) + f(x) \cdot \left(\frac{\mathrm{d}\;}{\mathrm{d}x} g(x)\right)
      $$
      </li>
<li>  The corresponding cases deals with the case where <tt>e</tt> is a quotient.  The
      <a href="https://en.wikipedia.org/wiki/Quotient_rule">quotient rule</a> is
      $$ \frac{\mathrm{d}\;}{\mathrm{d}x}\left(\frac{f(x)}{g(x)}\right) = 
         \frac{\displaystyle\left(\frac{\mathrm{d}\;}{\mathrm{d}x} f(x)\right)\cdot g(x) - 
         f(x) \cdot \left(\frac{\mathrm{d}\;}{\mathrm{d}x} g(x)\right)}{g(x) \cdot g(x)}
      $$
      </li>
<li>  The corresponding cases deals with the case where <tt>e</tt> is a power.  Now in order to take the derivative of an
      expression of the form
      $$  f(x)^{g(x)} $$
      we first need to rewrite this expression using the following trick:
      $$ f(x)^{g(x)} = \exp\bigl(\ln\bigl(f(x)^{g(x)}\bigr)\bigr) = \exp\bigl(g(x) \cdot \ln(f(x))\bigr) $$
      Then, we can recursively call <tt>diff</tt> for this expression.  This works, because the function
      <tt>diff</tt> can deal with both the exponential function $x \mapsto \exp(x)$ and with the natural
      logarithm $x \mapsto \ln(x)$.  This rewriting is done in The corresponding cases.
      </li>
<li>  The corresponding cases deals with the case where <tt>e</tt> has the form 
      $$\ln\bigl(f(x)\bigr)$$  
      In order to take the derivative of this expression, we first need to know the derivative of the natural
      logarithm.  This derivative is given as     
      $$ \frac{\mathrm{d}\;}{\mathrm{d}x} \ln(x) = \frac{1}{x}$$
      Then, using the <a href="https://en.wikipedia.org/wiki/Chain_rule">chain rule</a> we have that
      $$ \frac{\mathrm{d}\;}{\mathrm{d}x} \ln\bigl(f(x)\bigr) = \frac{\frac{\mathrm{d}\;}{\mathrm{d}x} f(x)}{f(x)}$$
      </li>
<li>  The corresponding cases deals with the case where <tt>e</tt> has the form $\exp\bigl(f(x)\bigr)$.  
      In order to take the derivative of this expression, we first need to know the derivative of the 
      <a href="https://en.wikipedia.org/wiki/Exponential_function">exponential function</a>.  
      This derivative is given as 
      $$ \frac{\mathrm{d}\;}{\mathrm{d}x} \exp(x) = \exp(x)$$    
      Then, using the <a href="https://en.wikipedia.org/wiki/Chain_rule">chain rule</a> we have that
      $$\frac{\mathrm{d}\;}{\mathrm{d}x} \exp\bigl(f(x)\bigr) = \left(\frac{\mathrm{d}\;}{\mathrm{d}x} f(x)\right) \cdot \exp\bigl(f(x)\bigr)
      $$
      </li>
<li>  The corresponding cases deals with the case where <tt>e</tt> is a variable and happens to be the same variable as
      <tt>x</tt>.   As we have
      $$\frac{\mathrm{d}x}{\mathrm{d}x} = 1,$$
      the function <tt>diff</tt> returns <tt>1</tt> in this case.
      </li>
<li>  Otherwise, the expression is assumed to be a constant and hence we return 0.
      </li>
</ol>

In [ ]:
let rec diff = function
    | Number _ -> Number 0
    | Variable -> Number 1
    | Binary(Add, f, g) -> Binary(Add, diff f, diff g)
    | Binary(Subtract, f, g) -> Binary(Subtract, diff f, diff g)
    | Binary(Multiply, f, g) -> Binary(Add, Binary(Multiply, diff f, g), Binary(Multiply, f, diff g))
    | Binary(Divide, f, g) ->
        Binary(Divide, Binary(Subtract, Binary(Multiply, diff f, g), Binary(Multiply, f, diff g)), Binary(Multiply, g, g))
    | Binary(Power, f, g) -> diff (Unary(Exp, Binary(Multiply, g, Unary(Ln, f))))
    | Unary(Ln, f) -> Binary(Divide, diff f, f)
    | Unary(Exp, f) -> Binary(Multiply, diff f, Unary(Exp, f))

The function below turns the typed AST into a readable expression and preserves precedence and associativity.

In [ ]:
let precedence = function Add | Subtract -> 1 | Multiply | Divide -> 2 | Power -> 3
let precedenceOp = function Binary(op, _, _) -> precedence op | _ -> 4

// Equal-precedence right children also need brackets for nonassociative operations.
let rec toString expr =
    let wrap needed child =
        let text = toString child
        if needed then "(" + text + ")" else text
    match expr with
    | Number n -> string n
    | Variable -> "x"
    | Unary(op, argument) ->
        (match op with Exp -> "exp" | Ln -> "ln") + "(" + toString argument + ")"
    | Binary(op, left, right) ->
        let p = precedence op
        let symbol = match op with Add -> " + " | Subtract -> " - " | Multiply -> "*" | Divide -> "/" | Power -> "**"
        let leftNeeds = precedenceOp left < p || (op = Power && precedenceOp left = p)
        let rightNeeds = precedenceOp right < p || (op <> Power && precedenceOp right = p)
        wrap leftNeeds left + symbol + wrap rightNeeds right

let rec evaluate x = function
    | Number n -> float n
    | Variable -> x
    | Binary(op, f, g) ->
        let a, b = evaluate x f, evaluate x g
        match op with Add -> a + b | Subtract -> a - b | Multiply -> a * b | Divide -> divide a b | Power -> a ** b
    | Unary(Exp, f) -> exp(evaluate x f)
    | Unary(Ln, f) -> log(evaluate x f)

In [ ]:
let test source =
    let expr = parse source
    let derivative = diff expr
    printfn "d/dx %s = %s" source (toString derivative)
    // A numerical derivative independently checks the symbolic rules.
    let x, h = 1.7, 1e-6
    let numerical = (evaluate (x + h) expr - evaluate (x - h) expr) / (2.0 * h)
    checkNotebook (abs(evaluate x derivative - numerical) < 1e-4)
    derivative

In [ ]:
test "x ** x" |> ignore

In [ ]:
test "ln(exp(x)) / exp(x/x)" |> ignore
let grouped = parse "x / (x * x)"
checkNotebook (parse (toString grouped) = grouped)

## Inspect the derivative tree

The same typed expression can be rendered as a tree to inspect product, chain and power rules.


In [ ]:
#load "../Support/Visualization.fsx"
open Visualization
let rec expressionTree = function
    | Number n -> Leaf(string n)
    | Variable -> Leaf "x"
    | Unary(op, expr) -> Node((match op with Exp -> "exp" | Ln -> "ln"), [expressionTree expr])
    | Binary(op, left, right) ->
        let label = match op with Add -> "+" | Subtract -> "-" | Multiply -> "*" | Divide -> "/" | Power -> "**"
        Node(label, [expressionTree left; expressionTree right])
let derivative = parse "x ** x" |> diff
let svg = derivative |> expressionTree |> treeToDot |> renderDot
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(svg), "text/html") |> ignore
